# Sponsor OS — Price v1: fit the reach model

Fits a small **NumPyro hierarchical model** of how many people each sponsorship
asset reaches, and caches posterior samples to Supabase (`pricing_posteriors`).
The Streamlit Tier Simulator and the deck generator only ever read that cache.

**Honesty rules (do not delete):**
1. Priors below are educated assumptions, clearly marked. With zero observations
   the model still runs and produces WIDE intervals — that is correct behavior,
   show them, never narrow them by hand.
2. The model estimates **impressions (value)**. Prices stay human-set.
3. The upload cell **refuses to run** if the fit did not converge
   (r_hat > 1.01 or any divergence). Don't fight the gate; fix the fit.

Run in **Google Colab** (free) or locally in an isolated venv. Never install
jax/numpyro into the Streamlit app environment.

In [ ]:
%pip install -q numpyro jax supabase python-dotenv matplotlib

## 1 — Observations (fill with REAL numbers)

- `footfall`: past fest gate counts (one number per past fest; rough is fine).
- `fest_reel_views`: view counts of reels posted **during past fest periods** —
  NOT off-season posts (off-season reach runs structurally lower and would bias
  the model pessimistic on the wrong regime).

Empty lists = priors-only mode (supported, intervals will be wide and labeled).

In [ ]:
OBSERVATIONS = {
    "footfall": [],          # e.g. [2800, 3500]
    "fest_reel_views": [],   # e.g. [5200, 8100, 4400]
}

# PRIORS — educated assumptions, NOT data. Sources: typical student-fest scale
# for a private-university chapter event + common event-marketing rules of
# thumb (booth visit rates ~20-30% of footfall, banner notice ~50-70%, etc.).
# Every value here is on display in the app's model card via observations_n.
PRIORS = {
    "footfall_guess": 3000.0,  # central guess for fest scale
    "footfall_sd": 0.7,        # ~ +/- a factor of 2 uncertainty
    # per-asset multiplier on fest_scale: (central, lognormal sd)
    "asset_multipliers": {
        "stage_logo": (1.2, 0.5),  # most attendees see the stage repeatedly
        "reel": (2.0, 0.8),        # fest reels travel beyond the gates
        "booth": (0.25, 0.6),      # fraction of footfall that visits a stall
        "banner": (0.6, 0.6),      # fraction that notices a printed banner
        "shoutout": (0.8, 0.7),    # story/stage mention reach vs fest scale
    },
}

import datetime
MODEL_VERSION = "v1-" + datetime.date.today().strftime("%Y%m%d")
N_OBS = len(OBSERVATIONS["footfall"]) + len(OBSERVATIONS["fest_reel_views"])
print(f"{MODEL_VERSION}: fitting with {N_OBS} real observation(s)")

## 2 — Model and MCMC

Latent `fest_scale` (anchored by footfall/reel observations when present)
times per-asset multipliers. Partial pooling: any real observation tightens
every asset's posterior through the shared scale.

In [ ]:
import numpyro
numpyro.set_host_device_count(2)

import jax
import jax.numpy as jnp
import numpy as np
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

ASSETS = list(PRIORS["asset_multipliers"])

def model(footfall_obs=None, reel_views_obs=None):
    fest_scale = numpyro.sample(
        "fest_scale",
        dist.LogNormal(jnp.log(PRIORS["footfall_guess"]), PRIORS["footfall_sd"]))
    mults = {}
    for asset in ASSETS:
        center, sd = PRIORS["asset_multipliers"][asset]
        mults[asset] = numpyro.sample(f"mult_{asset}", dist.LogNormal(jnp.log(center), sd))
    if footfall_obs is not None and len(footfall_obs):
        numpyro.sample("footfall", dist.LogNormal(jnp.log(fest_scale), 0.2),
                       obs=jnp.asarray(footfall_obs, dtype=jnp.float32))
    if reel_views_obs is not None and len(reel_views_obs):
        numpyro.sample("reel_views",
                       dist.LogNormal(jnp.log(fest_scale * mults["reel"]), 0.5),
                       obs=jnp.asarray(reel_views_obs, dtype=jnp.float32))

mcmc = MCMC(NUTS(model), num_warmup=1500, num_samples=2000, num_chains=2)
mcmc.run(jax.random.PRNGKey(0),
         footfall_obs=OBSERVATIONS["footfall"],
         reel_views_obs=OBSERVATIONS["fest_reel_views"])
mcmc.print_summary()
samples = {name: np.asarray(values) for name, values in mcmc.get_samples().items()}
impressions = {asset: samples["fest_scale"] * samples[f"mult_{asset}"] for asset in ASSETS}

In [ ]:
# Convergence diagnostics — the upload cell is GATED on these values.
from numpyro.diagnostics import summary as nps_summary

stats = nps_summary(mcmc.get_samples(group_by_chain=True))
R_HAT_MAX = float(max(float(np.max(site["r_hat"])) for site in stats.values()))
DIVERGENCES = int(np.sum(mcmc.get_extra_fields()["diverging"]))
print(f"r_hat_max = {R_HAT_MAX:.4f}  (gate: <= 1.01)")
print(f"divergences = {DIVERGENCES}  (gate: 0)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(ASSETS), figsize=(18, 3))
for axis, asset in zip(axes, ASSETS):
    axis.hist(impressions[asset], bins=50)
    p10, p90 = np.percentile(impressions[asset], [10, 90])
    axis.set_title(f"{asset}\n{p10:,.0f} - {p90:,.0f}")
plt.suptitle("Impressions per asset unit (posterior). Wide = honest.")
plt.tight_layout()
plt.show()

## 3 — Upload to Supabase (convergence-gated)

Refuses to ship garbage: if `r_hat_max > 1.01` or `DIVERGENCES > 0` this cell
raises instead of uploading. The override exists for emergencies only and
requires literally typing `I_UNDERSTAND` — if you find yourself doing that,
stop and ask why the fit is broken first.

In [ ]:
OVERRIDE = ""  # type I_UNDERSTAND to bypass the convergence gate (don't)

if (R_HAT_MAX > 1.01 or DIVERGENCES > 0) and OVERRIDE != "I_UNDERSTAND":
    raise SystemExit(
        f"REFUSING UPLOAD: r_hat_max={R_HAT_MAX:.4f}, divergences={DIVERGENCES}. "
        "The fit has not converged - increase warmup/samples or revisit priors.")

import datetime as _dt
import os
from getpass import getpass

try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
from supabase import create_client

url = os.environ.get("SUPABASE_URL") or getpass("SUPABASE_URL: ")
key = os.environ.get("SUPABASE_SERVICE_KEY") or getpass("SUPABASE_SERVICE_KEY: ")
client = create_client(url, key)

for asset in ASSETS:
    draws = [round(float(x), 1) for x in impressions[asset][:2000]]
    client.table("pricing_posteriors").upsert(
        {"model_version": MODEL_VERSION, "asset_type": asset,
         "posterior_samples": draws},
        on_conflict="model_version,asset_type").execute()

meta = {
    "model_version": MODEL_VERSION,
    "fitted_at": _dt.datetime.now(_dt.timezone.utc).isoformat(),
    "observations_n": N_OBS,
    "observations": OBSERVATIONS,
    "priors": {k: v for k, v in PRIORS.items()},
    "r_hat_max": R_HAT_MAX,
    "divergences": DIVERGENCES,
    "note": "Impressions-per-asset model. Prices stay human-set; this justifies value.",
}
client.table("pricing_posteriors").upsert(
    {"model_version": MODEL_VERSION, "asset_type": "_meta", "posterior_samples": meta},
    on_conflict="model_version,asset_type").execute()
print(f"Uploaded {len(ASSETS)} asset posteriors + _meta as {MODEL_VERSION}.")